In [1]:
import numpy as np
import pandas as pd

In [2]:
# Load dataset
dataset = pd.read_csv(r"C:\Users\HP\dataset.csv",)

In [3]:
# Drop irrelevant columns
dataset = dataset.drop([
    'Name',
    'contact_number',
    'Contact_email',
    'last_call_time',
    'references',
    'Sales',
    'Conversion Rate (%)'
], axis=1, errors='ignore')

In [4]:
# Convert date columns
date_cols = ['creation_date', 'first_call_attempt', 'last_call_date']

for col in date_cols:
    if col in dataset.columns:
        dataset[col] = pd.to_datetime(dataset[col], dayfirst=True)

In [5]:
# Fill missing lead_tag
if 'lead_tag' in dataset.columns:
    dataset['lead_tag'].fillna('not tagged', inplace=True)

C:\Users\HP\AppData\Local\Temp\ipykernel_6120\1225717337.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  dataset['lead_tag'].fillna('not tagged', inplace=True)


In [6]:
# Feature: was_contacted
dataset['was_contacted'] = np.where(
    dataset['first_call_attempt'].isnull(),
    0,
    1
)

In [7]:
# Feature: had_followup
dataset['had_followup'] = np.where(
    dataset['last_call_date'].isnull(),
    0,
    1
)

In [8]:
# Interaction hours
dataset['interaction_hours'] = (
    dataset['last_call_date'] - dataset['first_call_attempt']
).dt.total_seconds() / 3600

dataset['interaction_hours'] = dataset['interaction_hours'].fillna(0)

In [9]:
# First contact delay
dataset['first_contact_delay_hours'] = (
    dataset['first_call_attempt'] - dataset['creation_date']
).dt.total_seconds() / 3600

dataset['first_contact_delay_hours'] = dataset['first_contact_delay_hours'].fillna(-1)

In [10]:
# Lead age
dataset['lead_age'] = (
    dataset['last_call_date'] - dataset['creation_date']
).dt.total_seconds() / 3600

dataset['lead_age'] = dataset['lead_age'].fillna(-1)

In [11]:
# Create target variable
dataset['target'] = np.where(
    dataset['lead_stage'].str.lower() == 'converted',
    1,
    0
)

In [12]:
# Drop leakage + original columns
dataset = dataset.drop([
    'creation_date',
    'first_call_attempt',
    'last_call_date',
    'lead_stage'
], axis=1, errors='ignore')


In [13]:
# Remove lead_tag (data leakage)
if 'lead_tag' in dataset.columns:
    dataset = dataset.drop(['lead_tag'], axis=1)

print("Preprocessing completed")
print(dataset.head())

Preprocessing completed
              lead_status assigned_to            campaign_name   source_type  \
0  PENDING WITH EXECUTIVE     Anshad   Data Analytics Knovista     INSTAGRAM   
1  PENDING WITH EXECUTIVE     Anshad   Data Analytics Knovista  FACEBOOK_ADS   
2  PENDING WITH EXECUTIVE     Anshad   Data Analytics Knovista     INSTAGRAM   
3  PENDING WITH EXECUTIVE     Anshad   Data Analytics Knovista     INSTAGRAM   
4  PENDING WITH EXECUTIVE     Anshad   Data Analytics Knovista     INSTAGRAM   

    source_name  Total Leads  Cost per Lead  Total Cost  was_contacted  \
0     INSTAGRAM           45            9.0       405.0              0   
1  FACEBOOK_ADS           26           13.0       338.0              0   
2     INSTAGRAM            9           13.0       117.0              0   
3     INSTAGRAM           13            5.0        65.0              0   
4     INSTAGRAM           13            9.0       117.0              0   

   had_followup  interaction_hours  first_contact_

In [15]:
# Save cleaned dataset
dataset.to_csv("../data/cleaned_dataset.csv", index=False)